# Standalone Tuned Model Chat

This notebook is a cell-by-cell version of `src/chat_tuned.py`. It inlines the small runtime helpers from the repo so it can run without importing local `src/` modules.

Run the setup/config cells first, adjust `CONFIG`, then load the tokenizer/model. Use `chat_once(...)` for single prompts or `chat_loop()` for an interactive loop.

## 1. Imports And `.env` Loading

In [2]:
pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 KB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 15.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 KB 22.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
from __future__ import annotations

import os
import re
import shlex
from dataclasses import dataclass
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import __version__ as TRANSFORMERS_VERSION


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in (current, *current.parents):
        if (path / "src" / "chat_tuned.py").exists() or (path / ".git").exists():
            return path
    return current


REPO_ROOT = find_repo_root()
ENV_PATH = REPO_ROOT / ".env"
print(f"Repo root: {REPO_ROOT}")
print(f".env path: {ENV_PATH}")

Repo root: /lambda/nfs/azfs/llama32-local
.env path: /lambda/nfs/azfs/llama32-local/.env


In [4]:
_ENV_REF_RE = re.compile(r"\$\{(?P<braced>[A-Za-z_][A-Za-z0-9_]*)\}|\$(?P<bare>[A-Za-z_][A-Za-z0-9_]*)")


def expand_env_refs(value: str) -> str:
    def replace(match: re.Match[str]) -> str:
        name = match.group("braced") or match.group("bare")
        return os.environ.get(name, "")

    return _ENV_REF_RE.sub(replace, value)


def load_dotenv(path: Path = ENV_PATH) -> None:
    if not path.exists():
        return

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        try:
            parts = shlex.split(line, comments=True, posix=True)
        except ValueError:
            continue

        if not parts:
            continue
        if parts[0] == "export":
            parts = parts[1:]

        for part in parts:
            if "=" not in part:
                continue
            name, value = part.split("=", 1)
            if name:
                os.environ.setdefault(name, expand_env_refs(value))


def env_str(name: str, default: str = "") -> str:
    return os.environ.get(name, default)


def env_int(name: str, default: int) -> int:
    return int(env_str(name, str(default)))


def repo_path(path: str | Path) -> Path:
    path = Path(path).expanduser()
    return path if path.is_absolute() else REPO_ROOT / path


load_dotenv()
print("Loaded .env" if ENV_PATH.exists() else "No .env found; using process environment and defaults")

Loaded .env


## 2. Configuration

Edit this cell before loading the model. `checkpoint="auto"` uses the root model directory if it has `config.json`, otherwise the latest `checkpoint-*` directory below it.

In [5]:
@dataclass
class ChatConfig:
    model_path: str = env_str("DEFAULT_MODEL_PATH")
    checkpoint: str = "auto"  # auto, root, latest
    tokenizer_path: str | None = None
    backend: str = env_str("DEFAULT_BACKEND", "rocm:0")  # rocm:0, cuda:0, cpu
    dtype: str = env_str("DEFAULT_DTYPE", "auto")  # auto, bf16, fp16, fp32
    attention: str = env_str("DEFAULT_ATTENTION", "auto")  # auto, flash_attention_2, sdpa, default
    tf32: bool = True
    low_cpu_mem_usage: bool = True
    compile_model: bool = False
    max_new_tokens: int = env_int("MAX_NEW_TOKENS", 500)
    temperature: float = 0.0
    top_p: float = 0.9
    repetition_penalty: float = 1.1
    system_prompt: str = env_str("SYSTEM_PROMPT")


CONFIG = ChatConfig()
CONFIG

ChatConfig(model_path='~/azfs/model-root/model', checkpoint='auto', tokenizer_path=None, backend='cuda:0', dtype='auto', attention='auto', tf32=True, low_cpu_mem_usage=True, compile_model=False, max_new_tokens=500, temperature=0.0, top_p=0.9, repetition_penalty=1.1, system_prompt='identify the source of the question and answer the question based on the context provided.')

## 3. Runtime Helpers

In [6]:
_DTYPES = {
    "bf16": torch.bfloat16,
    "fp16": torch.float16,
    "fp32": torch.float32,
}


def transformers_dtype_kwarg() -> str:
    match = re.match(r"^(\d+)\.(\d+)", TRANSFORMERS_VERSION)
    if not match:
        return "dtype"

    major, minor = (int(part) for part in match.groups())
    return "dtype" if (major, minor) >= (4, 51) else "torch_dtype"


def checkpoint_step(path: Path) -> int:
    match = re.match(r"checkpoint-(\d+)$", path.name)
    return int(match.group(1)) if match else -1


def latest_checkpoint(model_dir: Path) -> Path | None:
    checkpoints = [
        path
        for path in model_dir.glob("checkpoint-*")
        if path.is_dir() and (path / "config.json").exists()
    ]
    return max(checkpoints, key=checkpoint_step) if checkpoints else None


def has_tokenizer_files(path: Path) -> bool:
    tokenizer_files = ("tokenizer.json", "tokenizer_config.json", "vocab.json", "spiece.model")
    return any((path / name).exists() for name in tokenizer_files)


def resolve_model_path(model_path: str, checkpoint: str) -> Path:
    if not model_path:
        raise ValueError("CONFIG.model_path is empty. Set it to a local model/checkpoint directory.")

    base = repo_path(model_path)
    print(f"Model base path: {base}")

    if checkpoint == "root":
        return base

    latest = latest_checkpoint(base)
    if checkpoint == "latest":
        if latest is None:
            raise FileNotFoundError(f"No checkpoint-* directory found under {base}")
        return latest

    if checkpoint != "auto":
        raise ValueError("checkpoint must be one of: auto, root, latest")
    if (base / "config.json").exists():
        return base
    if latest is not None:
        return latest

    raise FileNotFoundError(f"No model config found at {base} and no checkpoint-* directory exists below it.")


def resolve_tokenizer_path(model_path: Path, tokenizer_path: str | None) -> Path:
    if tokenizer_path is not None:
        return repo_path(tokenizer_path)

    if has_tokenizer_files(model_path):
        return model_path

    parent = model_path.parent
    if checkpoint_step(model_path) >= 0 and has_tokenizer_files(parent):
        return parent

    return model_path

In [7]:
def parse_backend(backend: str) -> tuple[str, int | None]:
    parts = backend.split(":", 1)
    name = parts[0].lower()

    if name == "cpu":
        return name, None
    if name in {"cuda", "rocm", "hip"}:
        return name, int(parts[1]) if len(parts) > 1 else 0

    raise ValueError(f"Unknown backend {backend!r}. Use rocm, rocm:<id>, cuda, cuda:<id>, or cpu.")


def resolve_dtype(dtype_name: str, backend_name: str) -> torch.dtype:
    if dtype_name != "auto":
        return _DTYPES[dtype_name]

    if backend_name == "cpu":
        return torch.float32
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def configure_accelerator(backend_name: str, device_id: int, tf32: bool) -> None:
    if not torch.cuda.is_available():
        raise RuntimeError(f"{backend_name.upper()} backend requested, but torch.cuda.is_available() is false.")
    if backend_name in {"rocm", "hip"} and torch.version.hip is None:
        raise RuntimeError(f"{backend_name.upper()} backend requested, but this PyTorch build is not ROCm/HIP.")

    torch.cuda.set_device(device_id)
    torch.backends.cuda.matmul.allow_tf32 = tf32
    torch.backends.cudnn.allow_tf32 = tf32
    torch.set_float32_matmul_precision("high" if tf32 else "highest")

    if hasattr(torch.backends.cuda, "enable_flash_sdp"):
        torch.backends.cuda.enable_flash_sdp(True)
    if hasattr(torch.backends.cuda, "enable_mem_efficient_sdp"):
        torch.backends.cuda.enable_mem_efficient_sdp(True)
    if hasattr(torch.backends.cuda, "enable_math_sdp"):
        torch.backends.cuda.enable_math_sdp(True)

    props = torch.cuda.get_device_properties(device_id)
    total_gb = props.total_memory / (1024**3)
    backend_label = "ROCm" if torch.version.hip else "CUDA"
    print(f"{backend_label} device {device_id}: {props.name} ({total_gb:.1f} GiB)")
    print(f"TF32 enabled: {tf32}")


def device_map_for(backend_name: str, device_id: int | None) -> dict:
    if backend_name == "cpu":
        return {"": "cpu"}
    return {"": device_id if device_id is not None else 0}

In [8]:
def load_tokenizer(tokenizer_path: Path):
    tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path), trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_model(model_path: Path, dtype: torch.dtype, device_map: dict, attention: str, low_cpu_mem_usage: bool):
    kwargs = {
        transformers_dtype_kwarg(): dtype,
        "device_map": device_map,
        "trust_remote_code": True,
        "low_cpu_mem_usage": low_cpu_mem_usage,
    }

    if attention == "default":
        print("Attention: default")
        return AutoModelForCausalLM.from_pretrained(str(model_path), **kwargs)

    if attention != "auto":
        print(f"Attention: {attention}")
        return AutoModelForCausalLM.from_pretrained(str(model_path), attn_implementation=attention, **kwargs)

    try:
        model = AutoModelForCausalLM.from_pretrained(str(model_path), attn_implementation="flash_attention_2", **kwargs)
        print("Attention: flash_attention_2")
        return model
    except (ImportError, TypeError, ValueError) as exc:
        print(f"Flash Attention 2 unavailable; trying SDPA. ({exc})")

    try:
        model = AutoModelForCausalLM.from_pretrained(str(model_path), attn_implementation="sdpa", **kwargs)
        print("Attention: sdpa")
        return model
    except (ImportError, TypeError, ValueError) as exc:
        print(f"SDPA unavailable; using default attention. ({exc})")

    print("Attention: default")
    return AutoModelForCausalLM.from_pretrained(str(model_path), **kwargs)


def input_device(model) -> torch.device:
    try:
        return model.device
    except AttributeError:
        return next(model.parameters()).device


def build_messages(system_prompt: str, user_prompt: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def format_messages(tokenizer, system_prompt: str, user_prompt: str) -> str:
    messages = build_messages(system_prompt, user_prompt)

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    return f"System: {system_prompt}\nUser: {user_prompt}\nAssistant:"

## 4. Load The Tokenizer And Model

This can take a while and may use a lot of VRAM/RAM.

In [9]:
backend_name, device_id = parse_backend(CONFIG.backend)
dtype = resolve_dtype(CONFIG.dtype, backend_name)

if backend_name in {"cuda", "rocm", "hip"}:
    configure_accelerator(backend_name, device_id or 0, CONFIG.tf32)

model_path = resolve_model_path(CONFIG.model_path, CONFIG.checkpoint)
tokenizer_path = resolve_tokenizer_path(model_path, CONFIG.tokenizer_path)
device_map = device_map_for(backend_name, device_id)

print(f"Model path: {model_path}")
print(f"Tokenizer path: {tokenizer_path}")
print(f"Backend: {CONFIG.backend}")
print(f"Device map: {device_map}")
print(f"Dtype: {dtype}")

tokenizer = load_tokenizer(tokenizer_path)
model = load_model(
    model_path=model_path,
    dtype=dtype,
    device_map=device_map,
    attention=CONFIG.attention,
    low_cpu_mem_usage=CONFIG.low_cpu_mem_usage,
)
model.eval()

if CONFIG.compile_model:
    print("Compiling model...")
    model = torch.compile(model)

print("Model ready")

CUDA device 0: NVIDIA A100-SXM4-40GB (39.5 GiB)
TF32 enabled: True
Model base path: /home/ubuntu/azfs/model-root/model


FileNotFoundError: No model config found at /home/ubuntu/azfs/model-root/model and no checkpoint-* directory exists below it.

## 5. Generate Responses

In [10]:
def generate_response(prompt: str, config: ChatConfig = CONFIG) -> str:
    text = format_messages(tokenizer, config.system_prompt, prompt)
    inputs = tokenizer(text, return_tensors="pt").to(input_device(model))
    input_tokens = inputs["input_ids"].shape[-1]

    do_sample = config.temperature > 0
    generate_kwargs = {
        "max_new_tokens": config.max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": config.repetition_penalty,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
    }
    if do_sample:
        generate_kwargs["temperature"] = config.temperature
        generate_kwargs["top_p"] = config.top_p

    with torch.inference_mode():
        outputs = model.generate(**inputs, **generate_kwargs)

    response = outputs[0][input_tokens:]
    return tokenizer.decode(response, skip_special_tokens=True).strip()


def chat_once(prompt: str, config: ChatConfig = CONFIG) -> str:
    response = generate_response(prompt, config)
    print(response)
    return response

In [11]:
prompt = "Write a short greeting from the tuned model."
response = chat_once(prompt)

NameError: name 'tokenizer' is not defined

## 6. Optional Interactive Loop

In [12]:
def chat_loop(config: ChatConfig = CONFIG) -> None:
    while True:
        prompt = input("\nPrompt> ").strip()
        if not prompt or prompt.lower() in {"exit", "quit"}:
            break
        print("\n" + generate_response(prompt, config))


# Uncomment to use the notebook like the CLI script.
# chat_loop()